导入模块

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from market_data import create_manager, today_str, load_tushare_token
from backtest import (
    build_market_matrices,
    simulate_risk_parity_backtest,
    calc_drawdown,
)

基础参数


In [ ]:
# ========= 基础配置 =========
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"

# ========= 数据管理器 =========
manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("manager ready")
print("db_path =", DB_PATH)

定义资产池

In [ ]:
# 你的第一版资产池示例，可按需要修改
WATCHLIST = [
    "510300.SH",  # 沪深300ETF
    "511010.SH",  # 国债ETF
    "518880.SH",  # 黄金ETF
]

WATCHLIST

读取本地市场数据

In [ ]:
START_DATE = "20180101"
END_DATE = today_str()

raw_prices = manager.store.get_daily_prices(
    ts_codes=WATCHLIST,
    start_date=START_DATE,
    end_date=END_DATE,
)

print(raw_prices.shape)
raw_prices.head()

检查数据完整性

In [ ]:
raw_prices.groupby("ts_code")["trade_date"].agg(["min", "max", "count"])

构建market字典

In [ ]:
market = build_market_matrices(
    data=raw_prices,
    fields=("open", "high", "low", "close", "amount"),
    date_col="trade_date",
    code_col="ts_code",
    date_format="%Y%m%d",
)

for k, v in market.items():
    print(k, v.shape)

查看close价格矩阵

In [ ]:
close_px = market["close"]
close_px.tail()

In [ ]:
设置回测参数

In [ ]:
BACKTEST_PARAMS = {
    "initial_cash": 1_000_000.0,
    "lookback_window": 120,
    "rebalance_freq": "Q",          # D / W / M / Q / Y
    "execution_price_type": "avg",  # open / close / high / low / avg
    "fee_rate_buy": 0.0005,
    "fee_rate_sell": 0.0005,
    "lot_size": 100,
    "max_trade_amount_ratio": 0.05,  # 单日成交额占比上限
    "amount_unit_scale": 1.0,
    "use_drift_trigger": False,
    "drift_threshold": 0.05,
    "risk_free_rate": 0.0,
    "annualization": 252,
}

RP_PREPARE_KWARGS = {
    "calendar": market["close"].index,
    "ffill": True,
    "ffill_limit": 5,
    "min_non_na_ratio": 0.8,
    "drop_all_na_dates": True,
}

RP_WEIGHT_KWARGS = {
    "method": "sample",      # sample / ewma
    "return_type": "log",    # log / simple
    "annualization": 252,
    "long_only": True,
    "drop_any_na": True,
}

运行回测

In [ ]:
result = simulate_risk_parity_backtest(
    market=market,
    initial_cash=BACKTEST_PARAMS["initial_cash"],
    lookback_window=BACKTEST_PARAMS["lookback_window"],
    rebalance_freq=BACKTEST_PARAMS["rebalance_freq"],
    execution_price_type=BACKTEST_PARAMS["execution_price_type"],
    fee_rate_buy=BACKTEST_PARAMS["fee_rate_buy"],
    fee_rate_sell=BACKTEST_PARAMS["fee_rate_sell"],
    lot_size=BACKTEST_PARAMS["lot_size"],
    max_trade_amount_ratio=BACKTEST_PARAMS["max_trade_amount_ratio"],
    amount_unit_scale=BACKTEST_PARAMS["amount_unit_scale"],
    use_drift_trigger=BACKTEST_PARAMS["use_drift_trigger"],
    drift_threshold=BACKTEST_PARAMS["drift_threshold"],
    rp_prepare_kwargs=RP_PREPARE_KWARGS,
    rp_weight_kwargs=RP_WEIGHT_KWARGS,
    risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
    annualization=BACKTEST_PARAMS["annualization"],
)

查看绩效汇总

In [ ]:
summary = result["summary"]
summary

查看净值曲线

In [ ]:
nav_df = result["nav_df"]
nav_df.tail()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(nav_df.index, nav_df["nav"], label="Risk Parity NAV")
plt.title("Portfolio NAV")
plt.xlabel("Date")
plt.ylabel("NAV")
plt.legend()
plt.grid(True)
plt.show()

查看回撤曲线

In [ ]:
drawdown = calc_drawdown(nav_df["nav"])

plt.figure(figsize=(12, 4))
plt.plot(drawdown.index, drawdown, label="Drawdown")
plt.title("Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.grid(True)
plt.legend()
plt.show()

查看组合日收益率

In [ ]:
returns = result["returns"]
returns.describe()

In [ ]:
查看目标权重

In [ ]:
target_weights_df = result["target_weights_df"]
target_weights_df.tail(10)

查看实际权重

In [ ]:
weights_df = result["weights_df"]
weights_df.tail(10)

查看持仓股数

In [ ]:
positions_df = result["positions_df"]
positions_df.tail(10)

In [ ]:
查看交易记录

In [ ]:
trades_df = result["trades_df"]
print(trades_df.shape)
trades_df.head(20)

按标的的汇总交易额

In [ ]:
if len(trades_df) > 0:
    trades_summary = trades_df.groupby(["ts_code", "side"])["trade_value"].sum().unstack(fill_value=0)
    trades_summary
else:
    print("no trades")

In [ ]:
查看调仓日志

In [ ]:
rebalance_log_df = result["rebalance_log_df"]
rebalance_log_df.head(20)

把关键结果打包成一个查看字典

In [ ]:
bt_view = {
    "summary": result["summary"],
    "nav_tail": result["nav_df"].tail(),
    "weights_tail": result["weights_df"].tail(),
    "positions_tail": result["positions_df"].tail(),
    "trades_tail": result["trades_df"].tail(),
    "rebalance_tail": result["rebalance_log_df"].tail(),
}

bt_view["summary"]

导出结果到本地文件

In [ ]:
EXPORT_DIR = "data/exports"

result["nav_df"].to_csv(f"{EXPORT_DIR}/nav.csv")
result["weights_df"].to_csv(f"{EXPORT_DIR}/weights.csv")
result["positions_df"].to_csv(f"{EXPORT_DIR}/positions.csv")
result["trades_df"].to_csv(f"{EXPORT_DIR}/trades.csv", index=False)
result["rebalance_log_df"].to_csv(f"{EXPORT_DIR}/rebalance_log.csv", index=False)

summary.to_csv(f"{EXPORT_DIR}/summary.csv")

print("export done")

快速切换成月度调仓

In [ ]:
result_monthly = simulate_risk_parity_backtest(
    market=market,
    initial_cash=1_000_000.0,
    lookback_window=120,
    rebalance_freq="M",
    execution_price_type="avg",
    fee_rate_buy=0.0005,
    fee_rate_sell=0.0005,
    lot_size=100,
    max_trade_amount_ratio=0.05,
    amount_unit_scale=1.0,
    use_drift_trigger=False,
    drift_threshold=0.05,
    rp_prepare_kwargs=RP_PREPARE_KWARGS,
    rp_weight_kwargs=RP_WEIGHT_KWARGS,
    risk_free_rate=0.0,
    annualization=252,
)

result_monthly["summary"]

快速切换成“季度调仓 + 偏离触发”

In [ ]:
result_q_drift = simulate_risk_parity_backtest(
    market=market,
    initial_cash=1_000_000.0,
    lookback_window=120,
    rebalance_freq="Q",
    execution_price_type="avg",
    fee_rate_buy=0.0005,
    fee_rate_sell=0.0005,
    lot_size=100,
    max_trade_amount_ratio=0.05,
    amount_unit_scale=1.0,
    use_drift_trigger=True,
    drift_threshold=0.08,
    rp_prepare_kwargs=RP_PREPARE_KWARGS,
    rp_weight_kwargs=RP_WEIGHT_KWARGS,
    risk_free_rate=0.0,
    annualization=252,
)

result_q_drift["summary"]

净值曲线对比

In [ ]:
nav_compare = pd.concat(
    [
        result["nav_df"]["nav"].rename("Q"),
        result_monthly["nav_df"]["nav"].rename("M"),
        result_q_drift["nav_df"]["nav"].rename("Q+Drift"),
    ],
    axis=1,
)

plt.figure(figsize=(12, 6))
for col in nav_compare.columns:
    plt.plot(nav_compare.index, nav_compare[col], label=col)

plt.title("NAV Comparison")
plt.xlabel("Date")
plt.ylabel("NAV")
plt.legend()
plt.grid(True)
plt.show()

最小回测入口函数

In [ ]:
def run_backtest(
    watchlist,
    start_date="20180101",
    end_date=None,
    initial_cash=1_000_000.0,
    lookback_window=120,
    rebalance_freq="Q",
    execution_price_type="avg",
    use_drift_trigger=False,
    drift_threshold=0.05,
):
    if end_date is None:
        end_date = today_str()

    raw_prices = manager.store.get_daily_prices(
        ts_codes=watchlist,
        start_date=start_date,
        end_date=end_date,
    )

    market = build_market_matrices(
        data=raw_prices,
        fields=("open", "high", "low", "close", "amount"),
        date_col="trade_date",
        code_col="ts_code",
        date_format="%Y%m%d",
    )

    result = simulate_risk_parity_backtest(
        market=market,
        initial_cash=initial_cash,
        lookback_window=lookback_window,
        rebalance_freq=rebalance_freq,
        execution_price_type=execution_price_type,
        fee_rate_buy=0.0005,
        fee_rate_sell=0.0005,
        lot_size=100,
        max_trade_amount_ratio=0.05,
        amount_unit_scale=1.0,
        use_drift_trigger=use_drift_trigger,
        drift_threshold=drift_threshold,
        rp_prepare_kwargs={
            "calendar": market["close"].index,
            "ffill": True,
            "ffill_limit": 5,
            "min_non_na_ratio": 0.8,
            "drop_all_na_dates": True,
        },
        rp_weight_kwargs={
            "method": "sample",
            "return_type": "log",
            "annualization": 252,
            "long_only": True,
            "drop_any_na": True,
        },
        risk_free_rate=0.0,
        annualization=252,
    )
    return result

调用最小入口

In [ ]:
result_simple = run_backtest(
    watchlist=WATCHLIST,
    start_date="20180101",
    initial_cash=1_000_000.0,
    lookback_window=120,
    rebalance_freq="Q",
    execution_price_type="avg",
    use_drift_trigger=False,
)

result_simple["summary"]